# Fundamentos de Ingeniería de Datos
## MODELADO: de la capa TRANSFORMED a un esquema estrella (capa SERVING)

```
TRANSFORMED                              SERVING (este notebook)
transformed_autos.parquet          →     serving.db (DuckDB)
(una tabla plana, 189,845 anuncios)      fact_listings + 6 dimensiones
```

| Capa | Dónde vive | Qué contiene | *Medallion* |
|---|---|---|---|
| RAW / EXTRACTED | `base_datos.db` (SQLite) | Datos como llegaron | Bronze |
| TRANSFORMED | `transformed_autos.parquet` | Datos limpios, **una tabla plana** | Silver |
| **SERVING** | **`serving.db` (DuckDB)** | **Esquema estrella: hechos + dimensiones** | **Gold** |

`serving.db` es la base que consultan el **tablero** y el **modelo** de la app de Streamlit.

### Configuración del entorno

In [1]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import (
    PROYECTO_BASE, PROCESSED_PATH, TRANSFORMACION_LOGS, MODELADO_PATH, SERVING_DB,
    MODELADO_REPORTES_PATH, ver_serving, tamano_mb, ruta_relativa
)

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {os.path.basename(PROYECTO_BASE)}/')
print()


utils.py importado correctamente
Proyecto base: ING_BD_ProjectF/



In [2]:
# ── Librerías, rutas y conexión ────────────────────────────────────────────

import json
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)

ETIQUETA       = 'autos'
PARQUET_ORIGEN = f'{PROCESSED_PATH}/transformed_{ETIQUETA}.parquet'           # origen: capa TRANSFORMED
LINEAGE_PATH   = f'{TRANSFORMACION_LOGS}/lineage_transformed_{ETIQUETA}.json'  # roles de las columnas
VALIDACION_CSV = f'{MODELADO_REPORTES_PATH}/validacion_serving_{ETIQUETA}.csv'

os.makedirs(MODELADO_PATH, exist_ok=True)
os.makedirs(MODELADO_REPORTES_PATH, exist_ok=True)

con = duckdb.connect(SERVING_DB)                    # si serving.db no existe, DuckDB lo crea

print()
print(f'Origen  : {ruta_relativa(PARQUET_ORIGEN)}')
print(f'Destino : {ruta_relativa(SERVING_DB)}')
print(f'DuckDB  : {duckdb.__version__}')
print()


Origen  : Transformacion_de_Datos/Processed/transformed_autos.parquet
Destino : Modelado_de_Datos/serving.db
DuckDB  : 2.0.1



---
## PASO 1 — Conocer la tabla plana

Creamos una **vista** (`VIEW`) sobre el Parquet: guarda solo la consulta, no los datos. Cada vez que usamos `base_autos`, DuckDB lee el archivo.

In [3]:
# ── Vista sobre el Parquet ─────────────────────────────────────────────────

con.execute(f'''
    CREATE OR REPLACE VIEW base_autos AS
    SELECT * FROM read_parquet('{PARQUET_ORIGEN}')
''')

con.sql('SELECT * FROM base_autos LIMIT 5').df()

,manufacturer,base_model,model,year,age,mileage,mileage_km,mileage_per_year,engine_liters,cylinders,turbo,transmission,drivetrain,fuel_type,mpg_city,mpg_highway,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,brand_country,brand_segment,price,price_drop,original_price,price_range,seller_name,seller_rating,driver_rating,driver_reviews_num,_batch_id,_extracted_at,_source_table,_row_hash,_transformed_at,_transform_version
0,Acura,ILX,ILX Hybrid 1.5L,2013,10,92945,149580,9294,1.5,4,False,Automatic,FWD,Gasoline,39.0,38.0,Black,Beige,False,False,False,Japan,Luxury,13988,300,14288,$1k-15k,Iconic Coach,NaN,4.4,12,1,2026-11-24 11:17:50,raw_autos,a5f41d2a94483372,2026-11-24 11:53:00,v1
1,Acura,ILX,ILX Hybrid 1.5L,2013,10,61539,99037,6154,1.5,4,False,CVT,FWD,Hybrid,39.0,38.0,Silver,Unknown,True,True,True,Japan,Luxury,16950,0,16950,$15k-30k,Napleton's Palatine Subaru,4.6,4.4,12,1,2026-11-24 11:17:50,raw_autos,d490101e1b4b1443,2026-11-24 11:53:00,v1
2,Acura,NSX,NSX Base,2017,6,15904,25595,2651,3.5,6,True,Automatic,AWD,Hybrid,21.0,22.0,White,Black,False,False,True,Japan,Luxury,139876,2111,141987,$100k-200k,Piazza Acura / Honda of Reading,4.7,4.8,17,1,2026-11-24 11:17:50,raw_autos,7497d846a2e6c6e0,2026-11-24 11:53:00,v1
3,Acura,NSX,NSX Base,2017,6,20946,33709,3491,3.5,6,True,Automatic,AWD,Hybrid,21.0,22.0,Black,Red,False,False,True,Japan,Luxury,149995,0,149995,$100k-200k,Ace Motorsports,NaN,4.8,17,1,2026-11-24 11:17:50,raw_autos,d6c522d9d7e07811,2026-11-24 11:53:00,v1
4,Acura,NSX,NSX Base,2017,6,21074,33915,3512,3.5,6,True,Automatic,AWD,Hybrid,21.0,22.0,Black,Red,True,True,True,Japan,Luxury,127991,0,127991,$100k-200k,Texas Hot Rides,3.6,4.8,17,1,2026-11-24 11:17:50,raw_autos,c45fc82ed381e2f2,2026-11-24 11:53:00,v1


In [4]:
# ── Columnas y tipos ───────────────────────────────────────────────────────

con.sql('DESCRIBE base_autos').df()[['column_name', 'column_type']]

,column_name,column_type
0,manufacturer,VARCHAR
1,base_model,VARCHAR
2,model,VARCHAR
3,year,BIGINT
4,age,BIGINT
5,mileage,BIGINT
6,mileage_km,BIGINT
7,mileage_per_year,BIGINT
8,engine_liters,DOUBLE
9,cylinders,BIGINT


In [5]:
# ── Roles de las columnas (definidos en TRANSFORMACION y guardados en el lineage) ──

with open(LINEAGE_PATH, encoding='utf-8') as f:
    ROLES = json.load(f)['roles_columnas']

pd.DataFrame([(rol, ', '.join(cols)) for rol, cols in ROLES.items()], columns=['rol', 'columnas'])

,rol,columnas
0,objetivo,price
1,caracteristicas_auto,"manufacturer, base_model, model, year, age, mi..."
2,solo_tablero_precio,"price_drop, original_price, price_range"
3,solo_tablero_info,"seller_name, seller_rating, driver_rating, dri..."
4,metadatos,"_batch_id, _extracted_at, _source_table, _row_..."


In [6]:
# ── Contenido actual de serving.db (ver_serving() está en utils.py) ────────

ver_serving(SERVING_DB)


  DuckDB: serving.db

  NOMBRE                         TIPO          FILAS   COLS
  ------------------------------ -------- ---------- ------
  dim_color                      tabla            15      2
  dim_history                    tabla             8      4
  dim_powertrain                 tabla           752      7
  dim_seller                     tabla        16,897      2
  dim_vehicle                    tabla         7,292      6
  dim_year                       tabla            24      2
  fact_listings                  tabla       189,845     21
  base_autos                     vista       189,845     37


---
## PASO 2 — Diseño lógico: ¿qué tablas necesitamos?

La tabla plana **repite** el mismo texto en muchas filas: "Ford", "F-150", "Automatic" aparecen miles de veces. Si un valor cambiara, habría que corregir miles de filas (*anomalía de actualización*). **Normalizar** es separar lo que se repite en su propia tabla, para guardarlo una sola vez.

`repeticion` = filas ÷ valores distintos: cuántas veces, en promedio, aparece cada valor.

In [7]:
# ── ¿Cuánto se repite cada columna? ────────────────────────────────────────

def repeticion(vista):
    columnas = [c[0] for c in con.sql(f'DESCRIBE {vista}').fetchall() if not c[0].startswith('_')]
    conteos  = ', '.join(f'COUNT(DISTINCT "{c}")' for c in columnas)
    filas, *distintos = con.sql(f'SELECT COUNT(*), {conteos} FROM {vista}').fetchall()[0]
    return pd.DataFrame({'columna': columnas, 'valores_distintos': distintos,
                         'repeticion': [round(filas / d, 1) if d else None for d in distintos]})

pd.set_option('display.max_rows', 100)
repeticion('base_autos').sort_values('repeticion', ascending=False)

,columna,valores_distintos,repeticion
10,turbo,2,94922.5
22,brand_segment,2,94922.5
18,accidents_or_damage,2,94922.5
19,one_owner,2,94922.5
20,personal_use_only,2,94922.5
11,transmission,3,63281.7
12,drivetrain,4,47461.2
26,price_range,5,37969.0
21,brand_country,6,31640.8
13,fuel_type,7,27120.7


### ¿Qué columna depende de cuál? (dependencias funcionales)

Una columna puede ir en una dimensión solo si **depende únicamente de la llave de esa dimensión**: para cada valor de la llave, un solo valor de la columna. Si no, al crear la dimensión la llave se repetiría.

In [8]:
# ── Dependencias funcionales: ¿cuántos grupos tienen más de un valor? ──────

def dependencia(llave, columna):
    return con.sql(f'''
        SELECT '{llave}' AS llave, '{columna}' AS columna,
               COUNT(*)                         AS grupos,
               SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS grupos_con_mas_de_un_valor
        FROM (SELECT {llave}, COUNT(DISTINCT {columna}) AS n FROM base_autos GROUP BY {llave})
    ''').df().astype({'grupos': 'int64', 'grupos_con_mas_de_un_valor': 'int64'})

pd.concat([
    dependencia('manufacturer, model',       'base_model'),
    dependencia('manufacturer',              'brand_country'),
    dependencia('manufacturer',              'brand_segment'),
    dependencia('year',                      'age'),
    dependencia('seller_name',               'seller_rating'),
    dependencia('manufacturer, model, year', 'driver_rating'),
], ignore_index=True)

,llave,columna,grupos,grupos_con_mas_de_un_valor
0,"manufacturer, model",base_model,7292,0
1,manufacturer,brand_country,30,0
2,manufacturer,brand_segment,30,0
3,year,age,24,0
4,seller_name,seller_rating,16897,2070
5,"manufacturer, model, year",driver_rating,21618,17


**Lectura:**

- `base_model`, `brand_country`, `brand_segment` y `age` **sí dependen** de su llave (0 grupos con más de un valor): pueden ir en `dim_vehicle` y `dim_year`.
- `seller_rating` **no depende** solo del vendedor: un mismo nombre aparece con varias calificaciones (sucursales distintas o calificaciones que cambiaron). Si fuera a `dim_seller`, `seller_name` se repetiría. Por eso queda en el hecho como **medida informativa**.
- `driver_rating` es la calificación de los conductores a un **modelo y año**, y tampoco es estrictamente única. También queda en el hecho como medida informativa.

### Diseño

**Grano:** una fila de `fact_listings` = **un anuncio de venta** (igual que la tabla plana).

| Columnas de la tabla plana | Se vuelven… | Concepto |
|---|---|---|
| `manufacturer`, `base_model`, `model`, `brand_country`, `brand_segment` | `dim_vehicle` | Dimensión **desnormalizada**: la marca va dentro del vehículo, sin tabla aparte |
| `engine_liters`, `cylinders`, `turbo`, `transmission`, `drivetrain`, `fuel_type` | `dim_powertrain` | Atributos técnicos que se repiten juntos |
| `year`, `age` | `dim_year` | Dimensión de tiempo: el año es la llave natural |
| `exterior_color`, `interior_color` | `dim_color` (usada **dos veces**) | **Dimensión de rol múltiple** (*role-playing*) |
| `accidents_or_damage`, `one_owner`, `personal_use_only` | `dim_history` | **Dimensión basura** (*junk*): 3 banderas sí/no en una tabla de 8 filas |
| `seller_name` | `dim_seller` | Informativa (solo tablero) |
| `price`, `price_drop`, `original_price`, `mileage`, `mileage_km`, `mileage_per_year`, `mpg_city`, `mpg_highway` | `fact_listings` (medidas) | Números que se suman o promedian |
| `seller_rating`, `driver_rating`, `driver_reviews_num` | `fact_listings` (medidas informativas) | Solo tablero; pueden ser nulas |
| `price_range`, `_row_hash` | `fact_listings` (atributos degenerados) | Describen al anuncio y no necesitan tabla propia |

**Todas** las columnas de negocio tienen un destino. De los metadatos técnicos solo pasa `_row_hash`, que basta para rastrear cada anuncio hasta TRANSFORMED, EXTRACTED y RAW; los demás se quedan en el Parquet.

---
## PASO 3 — Construir el esquema estrella (diseño físico)

```
                    dim_year           dim_history
                        │                   │
   dim_vehicle ──── fact_listings ──── dim_powertrain
                     │    │     │
     dim_color (exterior) │  dim_color (interior)
                          │
                      dim_seller
```

| Tipo de tabla | Qué guarda | Filas |
|---|---|---|
| **Hechos** (`fact_`) | Lo que se **mide**: precio, millaje, rendimiento | Muchas |
| **Dimensiones** (`dim_`) | El **contexto** para filtrar y agrupar: qué auto, qué motor, qué año… | Pocas |

**Diseño físico** — lo escribimos en cada `CREATE TABLE`:

| Elemento | Para qué |
|---|---|
| **Tipo de dato** (`INTEGER`, `VARCHAR`, `DOUBLE`, `BOOLEAN`) | Cada columna con su tipo definitivo |
| `PRIMARY KEY` | Identifica cada fila; no se repite |
| `UNIQUE` | La combinación de atributos de una dimensión no se repite |
| `ROW_NUMBER()` | Genera la llave numérica (**sustituta**) de cada dimensión |
| `REFERENCES` | Llave foránea: el hecho apunta a una dimensión que sí existe |

In [9]:
# ── Empezar de cero: crear serving.db nuevo ────────────────────────────────
# DuckDB no libera en disco el espacio de las tablas borradas con DROP TABLE: si el modelo se
# reconstruyera sobre el mismo archivo, serving.db crecería en cada ejecución. Por eso se borra
# el archivo completo y se vuelve a crear: el resultado es siempre el mismo (idempotencia).

con.close()
for archivo in (SERVING_DB, f'{SERVING_DB}.wal'):         # .wal: registro temporal de DuckDB
    if os.path.exists(archivo):
        os.remove(archivo)

con = duckdb.connect(SERVING_DB)
con.execute(f'''
    CREATE OR REPLACE VIEW base_autos AS
    SELECT * FROM read_parquet('{PARQUET_ORIGEN}')
''')

print()
print(f'{os.path.basename(SERVING_DB)} creado de nuevo (vacío)')
print('Listo para construir el modelo')
print()


serving.db creado de nuevo (vacío)
Listo para construir el modelo



### 3.1 `dim_vehicle`

Un vehículo se identifica por **marca + modelo con versión** (`UNIQUE (manufacturer, model)`). `base_model`, `brand_country` y `brand_segment` dependen de esa llave (PASO 2), así que van en la misma dimensión.

In [10]:
# ── dim_vehicle ────────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_vehicle (
        vehicle_id      INTEGER PRIMARY KEY,
        manufacturer    VARCHAR NOT NULL,
        base_model      VARCHAR NOT NULL,
        model           VARCHAR NOT NULL,
        brand_country   VARCHAR NOT NULL,
        brand_segment   VARCHAR NOT NULL,
        UNIQUE (manufacturer, model)
    )
''')

con.execute('''
    INSERT INTO dim_vehicle
    SELECT ROW_NUMBER() OVER (ORDER BY manufacturer, model) AS vehicle_id,
           manufacturer, base_model, model, brand_country, brand_segment
    FROM (SELECT DISTINCT manufacturer, base_model, model, brand_country, brand_segment FROM base_autos)
''')

con.sql('SELECT * FROM dim_vehicle LIMIT 5').df()

,vehicle_id,manufacturer,base_model,model,brand_country,brand_segment
0,1,Acura,CL,CL 3.2,Japan,Luxury
1,2,Acura,CL,CL 3.2 Type S,Japan,Luxury
2,3,Acura,ILX,ILX,Japan,Luxury
3,4,Acura,ILX,ILX 2.0L,Japan,Luxury
4,5,Acura,ILX,ILX 2.0L Technology,Japan,Luxury


### 3.2 `dim_powertrain`

El tren motriz: motor, transmisión, tracción y combustible. Hay cientos de combinaciones, pero se repiten miles de veces.

In [11]:
# ── dim_powertrain ─────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_powertrain (
        powertrain_id   INTEGER PRIMARY KEY,
        engine_liters   DOUBLE  NOT NULL,
        cylinders       INTEGER NOT NULL,
        turbo           BOOLEAN NOT NULL,
        transmission    VARCHAR NOT NULL,
        drivetrain      VARCHAR NOT NULL,
        fuel_type       VARCHAR NOT NULL,
        UNIQUE (engine_liters, cylinders, turbo, transmission, drivetrain, fuel_type)
    )
''')

con.execute('''
    INSERT INTO dim_powertrain
    SELECT ROW_NUMBER() OVER (ORDER BY fuel_type, engine_liters, cylinders, turbo, transmission, drivetrain) AS powertrain_id,
           engine_liters, cylinders, turbo, transmission, drivetrain, fuel_type
    FROM (SELECT DISTINCT engine_liters, cylinders, turbo, transmission, drivetrain, fuel_type FROM base_autos)
''')

con.sql('SELECT * FROM dim_powertrain LIMIT 5').df()

,powertrain_id,engine_liters,cylinders,turbo,transmission,drivetrain,fuel_type
0,1,1.6,4,True,Automatic,AWD,Diesel
1,2,1.6,4,True,Automatic,FWD,Diesel
2,3,1.9,4,True,Automatic,FWD,Diesel
3,4,2.0,4,False,Automatic,4WD,Diesel
4,5,2.0,4,False,Automatic,AWD,Diesel


### 3.3 `dim_year`

La llave es el propio año (llave **natural**). `age` (antigüedad al momento de los anuncios) depende solo del año.

In [12]:
# ── dim_year ───────────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_year (
        year    INTEGER PRIMARY KEY,
        age     INTEGER NOT NULL
    )
''')

con.execute('''
    INSERT INTO dim_year
    SELECT DISTINCT year, age FROM base_autos
''')

con.sql('SELECT * FROM dim_year ORDER BY year').df().T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
age,23,22,21,20,19,18,17,16,15,14,13,12,11,10,9,8,7,6,5,4,3,2,1,0


### 3.4 `dim_color` — dimensión de rol múltiple

Los colores exterior e interior usan **la misma lista de colores**. En lugar de dos dimensiones iguales, se crea **una sola**, y el hecho la referencia **dos veces** con dos llaves distintas (`exterior_color_id` e `interior_color_id`). A esto se le llama dimensión de **rol múltiple** (*role-playing dimension*).

In [13]:
# ── dim_color ──────────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_color (
        color_id    INTEGER PRIMARY KEY,
        color       VARCHAR NOT NULL UNIQUE
    )
''')

con.execute('''
    INSERT INTO dim_color
    SELECT ROW_NUMBER() OVER (ORDER BY color) AS color_id, color
    FROM (SELECT exterior_color AS color FROM base_autos
          UNION                                                  -- UNION quita repetidos
          SELECT interior_color FROM base_autos)
''')

con.sql('SELECT * FROM dim_color').df().T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
color_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
color,Beige,Black,Blue,Brown,Gold,Gray,Green,Orange,Other,Purple,Red,Silver,Unknown,White,Yellow


### 3.5 `dim_history` — dimensión basura

Tres banderas sí/no (`accidents_or_damage`, `one_owner`, `personal_use_only`) por separado serían tres dimensiones diminutas de 2 filas cada una. Una **dimensión basura** (*junk dimension*) junta todas sus combinaciones (2 × 2 × 2 = 8 filas) en una sola tabla, y el hecho guarda una sola llave.

In [14]:
# ── dim_history ────────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_history (
        history_id            INTEGER PRIMARY KEY,
        accidents_or_damage   BOOLEAN NOT NULL,
        one_owner             BOOLEAN NOT NULL,
        personal_use_only     BOOLEAN NOT NULL,
        UNIQUE (accidents_or_damage, one_owner, personal_use_only)
    )
''')

con.execute('''
    INSERT INTO dim_history
    SELECT ROW_NUMBER() OVER (ORDER BY accidents_or_damage, one_owner, personal_use_only) AS history_id,
           accidents_or_damage, one_owner, personal_use_only
    FROM (SELECT DISTINCT accidents_or_damage, one_owner, personal_use_only FROM base_autos)
''')

con.sql('SELECT * FROM dim_history').df()

,history_id,accidents_or_damage,one_owner,personal_use_only
0,1,False,False,False
1,2,False,False,True
2,3,False,True,False
3,4,False,True,True
4,5,True,False,False
5,6,True,False,True
6,7,True,True,False
7,8,True,True,True


### 3.6 `dim_seller`

El vendedor del anuncio. Es **informativa**: el tablero puede mostrarla, pero el modelo no la usa. Los anuncios sin vendedor ya llegaron como `'Unknown'` desde TRANSFORMACION (miembro desconocido).

In [15]:
# ── dim_seller ─────────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE dim_seller (
        seller_id     INTEGER PRIMARY KEY,
        seller_name   VARCHAR NOT NULL UNIQUE
    )
''')

con.execute('''
    INSERT INTO dim_seller
    SELECT ROW_NUMBER() OVER (ORDER BY seller_name) AS seller_id, seller_name
    FROM (SELECT DISTINCT seller_name FROM base_autos)
''')

con.sql('SELECT * FROM dim_seller LIMIT 5').df()

,seller_id,seller_name
0,1,#1 Auto Sales
1,2,#1 Brookville Chevrolet Buick
2,3,#1 Cochran Buick GMC Monroeville
3,4,#1 Cochran Buick GMC Robinson
4,5,#1 Cochran Chevrolet


### 3.7 `fact_listings` — el centro de la estrella

Guarda **llaves, medidas y atributos degenerados**. Cada llave se obtiene uniendo (`JOIN`) la tabla plana con su dimensión por los valores originales. `dim_color` se une **dos veces**, con dos alias distintos (`ce` exterior, `ci` interior).

`listing_id` se numera por `_row_hash`, así cada anuncio conserva el mismo número si el modelo se vuelve a construir con los mismos datos.

In [16]:
# ── fact_listings ──────────────────────────────────────────────────────────

con.execute('''
    CREATE TABLE fact_listings (

        listing_id           INTEGER PRIMARY KEY,

        vehicle_id           INTEGER NOT NULL REFERENCES dim_vehicle (vehicle_id),
        powertrain_id        INTEGER NOT NULL REFERENCES dim_powertrain (powertrain_id),
        year                 INTEGER NOT NULL REFERENCES dim_year (year),
        exterior_color_id    INTEGER NOT NULL REFERENCES dim_color (color_id),
        interior_color_id    INTEGER NOT NULL REFERENCES dim_color (color_id),
        history_id           INTEGER NOT NULL REFERENCES dim_history (history_id),
        seller_id            INTEGER NOT NULL REFERENCES dim_seller (seller_id),

        price                INTEGER NOT NULL,
        price_drop           INTEGER NOT NULL,
        original_price       INTEGER NOT NULL,
        mileage              INTEGER NOT NULL,
        mileage_km           INTEGER NOT NULL,
        mileage_per_year     INTEGER NOT NULL,
        mpg_city             DOUBLE  NOT NULL,
        mpg_highway          DOUBLE  NOT NULL,

        seller_rating        DOUBLE,
        driver_rating        DOUBLE,
        driver_reviews_num   INTEGER,

        price_range          VARCHAR NOT NULL,
        _row_hash            VARCHAR NOT NULL UNIQUE
    )
''')

con.execute('''
    INSERT INTO fact_listings
    SELECT ROW_NUMBER() OVER (ORDER BY b._row_hash) AS listing_id,

           v.vehicle_id,
           p.powertrain_id,
           b.year,
           ce.color_id                              AS exterior_color_id,
           ci.color_id                              AS interior_color_id,
           h.history_id,
           s.seller_id,

           b.price, b.price_drop, b.original_price,
           b.mileage, b.mileage_km, b.mileage_per_year,
           b.mpg_city, b.mpg_highway,

           b.seller_rating, b.driver_rating, b.driver_reviews_num,

           b.price_range,
           b._row_hash

    FROM base_autos AS b
    JOIN dim_vehicle    AS v  ON v.manufacturer = b.manufacturer AND v.model = b.model
    JOIN dim_powertrain AS p  ON p.engine_liters = b.engine_liters AND p.cylinders = b.cylinders
                             AND p.turbo = b.turbo AND p.transmission = b.transmission
                             AND p.drivetrain = b.drivetrain AND p.fuel_type = b.fuel_type
    JOIN dim_color      AS ce ON ce.color = b.exterior_color
    JOIN dim_color      AS ci ON ci.color = b.interior_color
    JOIN dim_history    AS h  ON h.accidents_or_damage = b.accidents_or_damage
                             AND h.one_owner = b.one_owner
                             AND h.personal_use_only = b.personal_use_only
    JOIN dim_seller     AS s  ON s.seller_name = b.seller_name
''')

con.sql('SELECT * FROM fact_listings LIMIT 5').df()

,listing_id,vehicle_id,powertrain_id,year,exterior_color_id,interior_color_id,history_id,seller_id,price,price_drop,original_price,mileage,mileage_km,mileage_per_year,mpg_city,mpg_highway,seller_rating,driver_rating,driver_reviews_num,price_range,_row_hash
0,1,3642,468,2016,7,2,2,11711,23726,1999,25725,83253,133983,11893,17.0,21.0,3.9,4.6,78,$15k-30k,000039eedfd8c53f
1,2,3554,644,2022,14,2,4,4812,79299,810,80109,23201,37338,23201,13.0,18.0,5.0,5.0,2,$50k-100k,00006db6a429610a
2,3,2530,489,2017,14,6,3,15695,23990,0,23990,116333,187220,19389,21.0,25.0,NaN,4.6,10,$15k-30k,0000a0b79de408cd
3,4,6577,469,2022,6,2,4,16127,41289,300,41589,6189,9960,6189,17.0,23.0,4.2,4.1,5,$30k-50k,0000a4c812d16f16
4,5,4235,450,2021,6,13,4,13678,43868,825,44693,48654,78301,24327,20.0,28.0,4.9,4.8,2,$30k-50k,0000dc7ad887d3dd


In [17]:
# ── Resultado: el esquema estrella en serving.db ───────────────────────────

con.sql('''
    SELECT table_name AS tabla, estimated_size AS filas_aprox, column_count AS columnas
    FROM duckdb_tables()
    ORDER BY table_name LIKE 'fact_%' DESC, table_name
''').df()

,tabla,filas_aprox,columnas
0,fact_listings,189845,21
1,dim_color,15,2
2,dim_history,8,4
3,dim_powertrain,752,7
4,dim_seller,16897,2
5,dim_vehicle,7292,6
6,dim_year,24,2


---
## PASO 4 — Validar el modelo

Al repartir los datos en tablas **no se debe perder ni duplicar nada**:

1. **Mismas filas** en el hecho que en la tabla plana.
2. **Mismas medidas**: la suma de precio, millaje y rebaja no cambia.
3. **0 huérfanas**: ningún anuncio apunta a una dimensión que no existe.
4. **Reconstrucción**: uniendo el hecho con sus dimensiones se recupera **exactamente** la tabla plana.

In [18]:
# ── 1 y 2. Filas y medidas: antes contra después ───────────────────────────

filas_medidas = con.sql('''
    SELECT 'filas'               AS validacion, (SELECT COUNT(*) FROM base_autos)        AS tabla_plana, (SELECT COUNT(*) FROM fact_listings)        AS esquema_estrella
    UNION ALL
    SELECT 'suma_price',               (SELECT SUM(price) FROM base_autos),               (SELECT SUM(price) FROM fact_listings)
    UNION ALL
    SELECT 'suma_mileage',             (SELECT SUM(mileage) FROM base_autos),             (SELECT SUM(mileage) FROM fact_listings)
    UNION ALL
    SELECT 'suma_price_drop',          (SELECT SUM(price_drop) FROM base_autos),          (SELECT SUM(price_drop) FROM fact_listings)
    UNION ALL
    SELECT 'seller_rating_no_nulos',   (SELECT COUNT(seller_rating) FROM base_autos),     (SELECT COUNT(seller_rating) FROM fact_listings)
    UNION ALL
    SELECT 'driver_rating_no_nulos',   (SELECT COUNT(driver_rating) FROM base_autos),     (SELECT COUNT(driver_rating) FROM fact_listings)
''').df().astype({'tabla_plana': 'int64', 'esquema_estrella': 'int64'})

filas_medidas['coincide'] = filas_medidas['tabla_plana'] == filas_medidas['esquema_estrella']
filas_medidas

,validacion,tabla_plana,esquema_estrella,coincide
0,filas,189845,189845,True
1,suma_price,6075021861,6075021861,True
2,suma_mileage,10585531210,10585531210,True
3,suma_price_drop,103032746,103032746,True
4,seller_rating_no_nulos,139583,139583,True
5,driver_rating_no_nulos,182666,182666,True


In [19]:
# ── 3. Registros huérfanos por llave foránea (todos deben ser 0) ───────────

huerfanas = con.sql('''
    SELECT
      SUM(CASE WHEN v.vehicle_id    IS NULL THEN 1 ELSE 0 END) AS huerfanas_vehicle,
      SUM(CASE WHEN p.powertrain_id IS NULL THEN 1 ELSE 0 END) AS huerfanas_powertrain,
      SUM(CASE WHEN y.year          IS NULL THEN 1 ELSE 0 END) AS huerfanas_year,
      SUM(CASE WHEN ce.color_id     IS NULL THEN 1 ELSE 0 END) AS huerfanas_exterior_color,
      SUM(CASE WHEN ci.color_id     IS NULL THEN 1 ELSE 0 END) AS huerfanas_interior_color,
      SUM(CASE WHEN h.history_id    IS NULL THEN 1 ELSE 0 END) AS huerfanas_history,
      SUM(CASE WHEN s.seller_id     IS NULL THEN 1 ELSE 0 END) AS huerfanas_seller
    FROM fact_listings AS f
    LEFT JOIN dim_vehicle    AS v  ON v.vehicle_id    = f.vehicle_id
    LEFT JOIN dim_powertrain AS p  ON p.powertrain_id = f.powertrain_id
    LEFT JOIN dim_year       AS y  ON y.year          = f.year
    LEFT JOIN dim_color      AS ce ON ce.color_id     = f.exterior_color_id
    LEFT JOIN dim_color      AS ci ON ci.color_id     = f.interior_color_id
    LEFT JOIN dim_history    AS h  ON h.history_id    = f.history_id
    LEFT JOIN dim_seller     AS s  ON s.seller_id     = f.seller_id
''').df().T.rename(columns={0: 'filas'}).astype('int64')

print()
print(f"Total de huérfanas: {int(huerfanas['filas'].sum())}")
print()

huerfanas


Total de huérfanas: 0



,filas
huerfanas_vehicle,0
huerfanas_powertrain,0
huerfanas_year,0
huerfanas_exterior_color,0
huerfanas_interior_color,0
huerfanas_history,0
huerfanas_seller,0


### 4. Reconstrucción de la tabla plana

Si el modelo es correcto, al unir el hecho con **todas** sus dimensiones se obtiene otra vez cada anuncio con todas sus columnas de negocio. `EXCEPT ALL` regresa las filas que están en una consulta y no en la otra: **en las dos direcciones debe dar 0**.

In [20]:
# ── 4. Reconstruir la tabla plana desde la estrella ────────────────────────

COLUMNAS_NEGOCIO = [c[0] for c in con.sql('DESCRIBE base_autos').fetchall() if not c[0].startswith('_')]

con.execute('''
    CREATE OR REPLACE TEMP VIEW reconstruida AS
    SELECT v.manufacturer, v.base_model, v.model, y.year, y.age,
           f.mileage, f.mileage_km, f.mileage_per_year,
           p.engine_liters, p.cylinders, p.turbo, p.transmission, p.drivetrain, p.fuel_type,
           f.mpg_city, f.mpg_highway,
           ce.color AS exterior_color, ci.color AS interior_color,
           h.accidents_or_damage, h.one_owner, h.personal_use_only,
           v.brand_country, v.brand_segment,
           f.price, f.price_drop, f.original_price, f.price_range,
           s.seller_name, f.seller_rating, f.driver_rating, f.driver_reviews_num
    FROM fact_listings AS f
    JOIN dim_vehicle    AS v  USING (vehicle_id)
    JOIN dim_powertrain AS p  USING (powertrain_id)
    JOIN dim_year       AS y  USING (year)
    JOIN dim_color      AS ce ON ce.color_id = f.exterior_color_id
    JOIN dim_color      AS ci ON ci.color_id = f.interior_color_id
    JOIN dim_history    AS h  USING (history_id)
    JOIN dim_seller     AS s  USING (seller_id)
''')

cols = ', '.join(COLUMNAS_NEGOCIO)
solo_plana       = con.sql(f'SELECT COUNT(*) FROM (SELECT {cols} FROM base_autos EXCEPT ALL SELECT {cols} FROM reconstruida)').fetchall()[0][0]
solo_reconstruida = con.sql(f'SELECT COUNT(*) FROM (SELECT {cols} FROM reconstruida EXCEPT ALL SELECT {cols} FROM base_autos)').fetchall()[0][0]

print()
print(f'Columnas comparadas                       : {len(COLUMNAS_NEGOCIO)}')
print(f'Filas en la tabla plana y no en la estrella: {solo_plana}')
print(f'Filas en la estrella y no en la tabla plana: {solo_reconstruida}')
print('Reconstrucción exacta' if solo_plana == solo_reconstruida == 0 else 'NO coincide: revisar el modelo')
print()


Columnas comparadas                       : 31
Filas en la tabla plana y no en la estrella: 0
Filas en la estrella y no en la tabla plana: 0
Reconstrucción exacta



In [21]:
# ── Guardar la validación ──────────────────────────────────────────────────

validacion = pd.concat([
    filas_medidas.rename(columns={'tabla_plana': 'esperado', 'esquema_estrella': 'obtenido'}),
    pd.DataFrame({'validacion': huerfanas.index, 'esperado': 0, 'obtenido': huerfanas['filas'].values}),
    pd.DataFrame({'validacion': ['reconstruccion_filas_distintas'], 'esperado': [0],
                  'obtenido': [solo_plana + solo_reconstruida]}),
], ignore_index=True)
validacion[['esperado', 'obtenido']] = validacion[['esperado', 'obtenido']].astype('int64')
validacion['coincide'] = validacion['esperado'] == validacion['obtenido']

validacion.to_csv(VALIDACION_CSV, index=False, encoding='utf-8-sig')

print()
print(f"¿Todas las validaciones correctas? {validacion['coincide'].all()}")
print(f'Guardada en: {ruta_relativa(VALIDACION_CSV)}')
print()

validacion


¿Todas las validaciones correctas? True
Guardada en: Modelado_de_Datos/Reportes/validacion_serving_autos.csv



,validacion,esperado,obtenido,coincide
0,filas,189845,189845,True
1,suma_price,6075021861,6075021861,True
2,suma_mileage,10585531210,10585531210,True
3,suma_price_drop,103032746,103032746,True
4,seller_rating_no_nulos,139583,139583,True
5,driver_rating_no_nulos,182666,182666,True
6,huerfanas_vehicle,0,0,True
7,huerfanas_powertrain,0,0,True
8,huerfanas_year,0,0,True
9,huerfanas_exterior_color,0,0,True


---
## PASO 5 — Consultar el modelo

Toda consulta sobre una estrella tiene la misma forma: **los hechos al centro, unidos a las dimensiones que pide la pregunta**. Estas son las consultas que usará el tablero.

In [22]:
# ── ¿Cuánto se deprecia un auto con los años, por segmento de marca? ───────

con.sql('''
    SELECT y.age                                   AS antiguedad,
           v.brand_segment,
           COUNT(*)                                AS anuncios,
           ROUND(MEDIAN(f.price))                  AS precio_mediano
    FROM fact_listings AS f
    JOIN dim_year    AS y USING (year)
    JOIN dim_vehicle AS v USING (vehicle_id)
    WHERE y.age <= 10
    GROUP BY y.age, v.brand_segment
    ORDER BY v.brand_segment, y.age
''').df().pivot(index='antiguedad', columns='brand_segment', values='precio_mediano')

brand_segment,Luxury,Mainstream
antiguedad,,
0,64995.0,41836.0
1,51990.0,39996.0
2,46146.0,33800.0
3,39992.0,27500.0
4,35882.0,25995.0
5,32408.0,23239.0
6,28267.0,20495.0
7,25495.0,17998.0
8,20998.0,16725.0


In [23]:
# ── ¿Cuánto vale un modelo según su millaje? (ejemplo: Toyota RAV4) ────────

con.sql('''
    SELECT CASE WHEN f.mileage <  30000 THEN '1. < 30k'
                WHEN f.mileage <  60000 THEN '2. 30k-60k'
                WHEN f.mileage < 100000 THEN '3. 60k-100k'
                ELSE                         '4. 100k+' END AS millaje,
           COUNT(*)                                          AS anuncios,
           ROUND(MEDIAN(f.price))                            AS precio_mediano
    FROM fact_listings AS f
    JOIN dim_vehicle AS v USING (vehicle_id)
    WHERE v.manufacturer = 'Toyota' AND v.base_model = 'RAV4'
    GROUP BY millaje
    ORDER BY millaje
''').df()

,millaje,anuncios,precio_mediano
0,1. < 30k,624,36835.0
1,2. 30k-60k,628,31080.0
2,3. 60k-100k,454,26316.0
3,4. 100k+,213,15175.0


In [24]:
# ── ¿Cuánto baja el precio un accidente o tener varios dueños? ─────────────

con.sql('''
    SELECT h.accidents_or_damage, h.one_owner,
           COUNT(*)                     AS anuncios,
           ROUND(MEDIAN(f.price))       AS precio_mediano,
           ROUND(MEDIAN(y.age), 1)      AS antiguedad_mediana
    FROM fact_listings AS f
    JOIN dim_history AS h USING (history_id)
    JOIN dim_year    AS y USING (year)
    GROUP BY h.accidents_or_damage, h.one_owner
    ORDER BY h.accidents_or_damage, h.one_owner DESC
''').df()

,accidents_or_damage,one_owner,anuncios,precio_mediano,antiguedad_mediana
0,False,True,88781,32999.0,3.0
1,False,False,57315,24000.0,5.0
2,True,True,18152,26500.0,4.0
3,True,False,25597,18529.0,7.0


In [25]:
# ── Rol múltiple en acción: combinaciones de color exterior e interior ─────

con.sql('''
    SELECT ce.color AS exterior, ci.color AS interior,
           COUNT(*) AS anuncios,
           ROUND(MEDIAN(f.price)) AS precio_mediano
    FROM fact_listings AS f
    JOIN dim_color AS ce ON ce.color_id = f.exterior_color_id
    JOIN dim_color AS ci ON ci.color_id = f.interior_color_id
    GROUP BY ce.color, ci.color
    ORDER BY anuncios DESC
    LIMIT 8
''').df()

,exterior,interior,anuncios,precio_mediano
0,Black,Black,25992,29900.0
1,White,Black,24155,30595.0
2,Gray,Black,18424,27999.0
3,Silver,Black,13969,25293.0
4,Red,Black,8557,26998.0
5,Blue,Black,8253,26849.0
6,White,Gray,7552,26264.0
7,Gray,Gray,6368,23998.0


In [26]:
# ── Cerrar la conexión ─────────────────────────────────────────────────────

con.execute('DROP VIEW IF EXISTS base_autos')     # la vista apunta a una ruta local: no se deja en serving.db
con.execute('CHECKPOINT')                         # termina de escribir en el archivo
con.close()

print()
print(f'Modelo guardado en: {ruta_relativa(SERVING_DB)}  ({tamano_mb(SERVING_DB):.1f} MB)')
print()


Modelo guardado en: Modelado_de_Datos/serving.db  (35.8 MB)



In [27]:
ver_serving(SERVING_DB)


  DuckDB: serving.db

  NOMBRE                         TIPO          FILAS   COLS
  ------------------------------ -------- ---------- ------
  dim_color                      tabla            15      2
  dim_history                    tabla             8      4
  dim_powertrain                 tabla           752      7
  dim_seller                     tabla        16,897      2
  dim_vehicle                    tabla         7,292      6
  dim_year                       tabla            24      2
  fact_listings                  tabla       189,845     21


### Copia para la app

La app de Streamlit debe funcionar en cualquier computadora con solo su carpeta, así que lleva **su propia copia** de `serving.db`. Se copia aquí, al terminar el modelo, para que la app siempre tenga la versión más reciente.

In [28]:
# ── Copiar serving.db a la carpeta de la app ───────────────────────────────

import shutil
from utils import STREAMLIT_PATH

os.makedirs(STREAMLIT_PATH, exist_ok=True)
copia = shutil.copy2(SERVING_DB, f'{STREAMLIT_PATH}/serving.db')

print()
print(f'{ruta_relativa(SERVING_DB)}  →  {ruta_relativa(copia)}  ({tamano_mb(copia):.1f} MB)')
print()


Modelado_de_Datos/serving.db  →  streamlit/serving.db  (35.8 MB)



> La vista `base_autos` se borra al final: guarda la ruta del Parquet en esta computadora, y `serving.db` debe funcionar en cualquier otra (por ejemplo, copiado dentro de la carpeta de la app). Las 7 tablas sí guardan los datos.

---
### Resumen

| Paso | Resultado |
|---|---|
| 1 | Vista `base_autos` sobre el Parquet y roles de las columnas desde el lineage |
| 2 | Diseño lógico: repetición por columna y dependencias funcionales |
| 3 | Esquema estrella en `serving.db`: `fact_listings` + 6 dimensiones |
| 4 | Mismas filas, mismas medidas, 0 huérfanas y reconstrucción exacta de la tabla plana |
| 5 | Consultas hechos + dimensiones para el tablero y copia de `serving.db` a `streamlit/` |

| Tabla | Tipo | Llave | Concepto |
|---|---|---|---|
| `fact_listings` | Hechos | `listing_id` | Grano: un anuncio de venta |
| `dim_vehicle` | Dimensión | `vehicle_id` | Desnormalizada (incluye la marca) |
| `dim_powertrain` | Dimensión | `powertrain_id` | Atributos técnicos |
| `dim_year` | Dimensión | `year` | Llave natural |
| `dim_color` | Dimensión | `color_id` | Rol múltiple (exterior e interior) |
| `dim_history` | Dimensión | `history_id` | Dimensión basura |
| `dim_seller` | Dimensión | `seller_id` | Informativa |

**Siguiente paso:** la app de Streamlit consulta `serving.db` uniendo `fact_listings` con sus dimensiones.